# CSPF — consistency-regularised fine-tuning (does the corpus make models more robust?)
**Question.** Does training on CSPF pairs with a consistency loss reduce the flip rate on *held-out* reviews, without hurting accuracy?
**Design (all choices fixed in advance; report whatever comes out).**
- Split the 48k source reviews by `dataset::review_idx` into train 75% / dev 5% / test 20% (fixed split seed). All pairs of a review stay on the same side, so test pairs come from unseen reviews.
- One backbone (`distilroberta-base` by default; `roberta-base` if time allows), same hyper-parameters for every method, 3 training seeds.
- Methods:
  - `erm` – cross-entropy on original texts only (baseline).
  - `aug_prop` – plus cross-entropy on perturbed texts of the *propositional* families (P2, P8, P10, P11, P12) with the original's gold label.
  - `cr_prop` – plus symmetric KL consistency between the predictions on original and perturbed text, propositional families only.
  - `cr_all` – consistency on all 12 families (ablation: forcing invariance on meaning-modulating families such as negation and sarcasm *should* hurt; this checks it).
  - `aug_all` – cross-entropy on perturbed texts of all families (noisy labels; ablation).
- Evaluation on test pairs (families capped at 6,000 pairs, re-weighted to corpus proportions): accuracy on originals and perturbed texts, flip rate FR overall, FR on propositional / modulating families, FR on families **not used for training** by the method (generalisation), cluster-bootstrap CI of ΔFR versus `erm`.
- **Runtime (T4):** ≈ 5–6 min per run with distilroberta (15 runs ≈ 1.5 h); resumable, each run's result is saved to Drive.
- Uses only the released corpus (gold labels `sentiment` of the source datasets). No API.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
OUT_DIR   = f"{BASE}/results_FIXED/consistency"
BACKBONE  = "distilroberta-base"
METHODS   = ["erm", "aug_prop", "cr_prop", "cr_all", "aug_all"]
SEEDS     = [0, 1, 2]
LAMBDA    = 1.0                  # weight of the consistency / augmentation term
EPOCHS, BS, LR, MAXLEN, WARMUP = 2, 32, 2e-5, 96, 0.06
SPLIT_SEED, EVAL_CAP = 0, 6000
PROP = ["P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"]
import os; os.makedirs(OUT_DIR+"/runs", exist_ok=True)

In [ ]:
import pandas as pd, numpy as np, os, glob, time, warnings; warnings.filterwarnings("ignore")
cols = ["source_dataset","review_idx","original_review","perturbed_review","sentiment","perturbation_family"]
df = pd.read_csv(FIXED_CSV, usecols=cols, low_memory=False)
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str)
df["y"]   = (df.sentiment=="positive").astype(np.int64)
FAMS = sorted(df.perturbation_family.unique())
U = df.drop_duplicates("key")[["key","source_dataset","original_review","y"]].reset_index(drop=True)
assert U.y.isin([0,1]).all() and (df.groupby("key").y.nunique()==1).all()
rng = np.random.default_rng(SPLIT_SEED); perm = rng.permutation(len(U)); n=len(U)
U["split"] = "train"; U.loc[perm[:int(.20*n)],"split"] = "test"; U.loc[perm[int(.20*n):int(.25*n)],"split"] = "dev"
df["split"] = df.key.map(U.set_index("key").split)
print(len(df),"pairs |",n,"reviews |", U.split.value_counts().to_dict())
print(df.groupby("split").perturbation_family.value_counts().unstack(0).fillna(0).astype(int))

In [ ]:
# ── fixed evaluation set (identical for every run): all test originals + capped, re-weighted test pairs ──
test_pairs = df[df.split=="test"]
sh = test_pairs.sample(frac=1, random_state=SPLIT_SEED)
EP = sh[sh.groupby("perturbation_family").cumcount() < EVAL_CAP].sort_index().reset_index(drop=True)
fam_tot = test_pairs.groupby("perturbation_family").size(); fam_n = EP.groupby("perturbation_family").size()
EP["w"] = EP.perturbation_family.map(fam_tot/fam_n).astype(float)                     # corpus-proportional weights
EO = U[U.split=="test"].reset_index(drop=True)
key2o = pd.Series(np.arange(len(EO)), index=EO.key); EP["o_idx"] = EP.key.map(key2o).values
print(len(EO),"test originals;",len(EP),"evaluation pairs (weights sum =",round(EP.w.sum()),")")

# ── training pools: for every train review, the pair indices usable by each method ──
TR = U[U.split=="train"].reset_index(drop=True)
trp = df[df.split=="train"].reset_index(drop=True)
def pools(fams):
    sub = trp[trp.perturbation_family.isin(fams)]
    return {k: v for k, v in sub.groupby("key").indices.items()}, sub.perturbed_review.values
POOLS = {"prop": pools(PROP), "all": pools(FAMS)}
print("train reviews:", len(TR), "| with a propositional pair:", len(POOLS["prop"][0]), "| with any pair:", len(POOLS["all"][0]))
METHOD_CFG = {   # name: (pool, loss on perturbed)
  "erm": (None, None), "aug_prop": ("prop","ce"), "cr_prop": ("prop","kl"), "cr_all": ("all","kl"), "aug_all": ("all","ce")}

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
dev = "cuda" if torch.cuda.is_available() else "cpu"; print(dev)
tok = AutoTokenizer.from_pretrained(BACKBONE)

def enc(texts): return tok(list(texts), padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)

@torch.no_grad()
def predict(model, texts, bs=256):
    model.eval(); out = np.zeros(len(texts), dtype=np.int64); order = np.argsort([len(t) for t in texts])
    for i in range(0, len(order), bs):
        ix = order[i:i+bs]
        with torch.autocast(dev, dtype=torch.float16, enabled=(dev=="cuda")):
            out[ix] = model(**enc([texts[j] for j in ix])).logits.argmax(-1).cpu().numpy()
    return out

def sym_kl(lp, lq):   # log-prob inputs, per-example symmetric KL
    p, q = lp.exp(), lq.exp()
    return 0.5*((p*(lp-lq)).sum(-1) + (q*(lq-lp)).sum(-1))

def train(method, seed):
    torch.manual_seed(seed); rs = np.random.default_rng(seed)
    pool_name, ploss = METHOD_CFG[method]
    model = AutoModelForSequenceClassification.from_pretrained(BACKBONE, num_labels=2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(TR)/BS)); sch = get_linear_schedule_with_warmup(opt, int(WARMUP*steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev=="cuda"))
    pool, ptexts = POOLS[pool_name] if pool_name else ({}, None)
    texts, ys, keys = TR.original_review.values, TR.y.values, TR.key.values
    for ep in range(EPOCHS):
        model.train(); order = rs.permutation(len(TR)); t0 = time.time(); tot = 0.0
        for s in range(0, len(order), BS):
            ix = order[s:s+BS]; x = [texts[i] for i in ix]; y = torch.tensor(ys[ix], device=dev)
            if pool_name:   # one random usable perturbation per review (mask = review has one)
                pj = [(pool[keys[i]][rs.integers(len(pool[keys[i]]))] if keys[i] in pool else -1) for i in ix]
                m = np.array([j >= 0 for j in pj]); xp = [ptexts[j] if j >= 0 else texts[i] for i, j in zip(ix, pj)]
            with torch.autocast(dev, dtype=torch.float16, enabled=(dev=="cuda")):
                if pool_name:
                    lg = model(**enc(x + xp)).logits.float(); lo, lp = lg[:len(x)], lg[len(x):]
                    mt = torch.tensor(m, device=dev, dtype=torch.float32); ce = F.cross_entropy(lo, y)
                    if ploss == "ce":  extra = (F.cross_entropy(lp, y, reduction="none")*mt).sum()/mt.sum().clamp(min=1)
                    else:              extra = (sym_kl(F.log_softmax(lo,-1), F.log_softmax(lp,-1))*mt).sum()/mt.sum().clamp(min=1)
                    loss = ce + LAMBDA*extra
                else:
                    loss = F.cross_entropy(model(**enc(x)).logits.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
        print(f"  {method} seed{seed} epoch {ep+1}: loss {tot/np.ceil(len(order)/BS):.4f} ({time.time()-t0:.0f}s)")
    return model

In [ ]:
def evaluate(pred_o, pred_p):
    # pred_o: predictions on EO (test originals); pred_p: predictions on EP.perturbed_review
    corr_o = (pred_o == EO.y.values)
    po_pair = pred_o[EP.o_idx.values]                       # original prediction aligned to each pair
    flip = (po_pair != pred_p).astype(np.int8); oc = corr_o[EP.o_idx.values]
    w = EP.w.values; fam = EP.perturbation_family.values
    def wm(x, mask=None):
        mask = np.ones(len(x), bool) if mask is None else mask
        return float((x[mask]*w[mask]).sum()/w[mask].sum())
    propm = np.isin(fam, PROP)
    r = dict(acc_orig=float(corr_o.mean()), acc_pert=wm((pred_p == EP.y.values).astype(float)), FR=wm(flip), FRc=wm(flip, oc), FRw=wm(flip, ~oc),
             FR_prop=wm(flip, propm), FR_mod=wm(flip, ~propm))
    for f in FAMS: r["FR_"+f] = wm(flip, fam==f)
    r["DR"] = wm(((flip==1)&oc).astype(float)); r["IRR"] = wm(((flip==1)&~oc).astype(float))
    return r, flip

def run_one(method, seed):
    f = f"{OUT_DIR}/runs/{method}_s{seed}.npz"
    if os.path.exists(f): return
    t0 = time.time(); model = train(method, seed)
    pred_o = predict(model, EO.original_review.tolist()); pred_p = predict(model, EP.perturbed_review.tolist())
    dv = U[U.split=="dev"]; acc_dev = float((predict(model, dv.original_review.tolist()) == dv.y.values).mean())
    r, flip = evaluate(pred_o, pred_p); r.update(method=method, seed=seed, acc_dev=acc_dev, minutes=(time.time()-t0)/60)
    np.savez_compressed(f, flip=flip, pred_o=pred_o.astype(np.int8), **{k: np.array(v) for k, v in r.items()})
    print(f"[{method} s{seed}] acc_o {r['acc_orig']:.4f} acc_p {r['acc_pert']:.4f} FR {r['FR']:.4f} FR_prop {r['FR_prop']:.4f} FR_mod {r['FR_mod']:.4f}  ({r['minutes']:.1f} min)")
    del model; torch.cuda.empty_cache()

In [ ]:
# RUN (resumable: finished runs are skipped after a disconnect). Order = seed-major so every method has seed 0 first.
for seed in SEEDS:
    for method in METHODS:
        run_one(method, seed)

In [ ]:
# ── summary: mean ± sd over seeds, generalisation to families NOT used in training ──
R = []
for f in sorted(glob.glob(f"{OUT_DIR}/runs/*.npz")):
    z = np.load(f, allow_pickle=True); R.append({k: z[k].item() for k in z.files if k not in ("flip","pred_o")})
R = pd.DataFrame(R)
w = EP.w.values; fam = EP.perturbation_family.values
TRAINED = {"erm": [], "aug_prop": PROP, "cr_prop": PROP, "cr_all": FAMS, "aug_all": FAMS}
FL = {(r.method, int(r.seed)): np.load(f"{OUT_DIR}/runs/{r.method}_s{int(r.seed)}.npz")["flip"] for r in R.itertuples()}
def unseen_fr(row):
    m = ~np.isin(fam, TRAINED[row.method]); fl = FL[(row.method, int(row.seed))]
    return float((fl[m]*w[m]).sum()/w[m].sum()) if m.any() else np.nan
R["FR_unseen_families"] = [unseen_fr(r) for r in R.itertuples()]
show = ["acc_orig","acc_pert","FR","FRc","FRw","FR_prop","FR_mod","FR_unseen_families","DR","IRR"]
S = R.groupby("method")[show].agg(["mean","std"]).round(4).loc[[m for m in METHODS if m in R.method.unique()]]
print(S.to_string()); S.to_csv(f"{OUT_DIR}/summary_mean_sd.csv"); R.to_csv(f"{OUT_DIR}/all_runs.csv", index=False)
Fm = R.groupby("method")[[c for c in R.columns if c.startswith("FR_P")]].mean().round(4).T; print("\n", Fm.to_string()); Fm.to_csv(f"{OUT_DIR}/family_FR_by_method.csv")

In [ ]:
# ΔFR (method − erm): average the flip indicators over seeds, then cluster-bootstrap over test reviews (same pairs for all methods)
rng = np.random.default_rng(7); keys_, inv = np.unique(EP.key.values, return_inverse=True); nk = len(keys_)
def mean_flip(method): return np.mean([FL[(method, s)] for s in SEEDS if (method, s) in FL], axis=0)
base = mean_flip("erm"); rows = []
for m in [x for x in METHODS if x != "erm" and any((x, s) in FL for s in SEEDS)]:
    d = mean_flip(m) - base
    for name, mask in [("all families", np.ones(len(d), bool)), ("propositional families", np.isin(fam, PROP)), ("modulating families", ~np.isin(fam, PROP)),
                       ("families not trained on", ~np.isin(fam, TRAINED[m]))]:
        if not mask.any(): continue
        ww = w*mask; est = (ww*d).sum()/ww.sum(); vals = []
        for _ in range(1000):
            cnt = np.bincount(rng.integers(0, nk, nk), minlength=nk)[inv]; a = ww*cnt
            vals.append((a*d).sum()/a.sum())
        lo, hi = np.percentile(vals, [2.5, 97.5]); rows.append(dict(method=m, subset=name, delta_FR=round(est,4), ci_lo=round(lo,4), ci_hi=round(hi,4), sig=bool(lo>0 or hi<0)))
D = pd.DataFrame(rows); print(D.to_string(index=False)); D.to_csv(f"{OUT_DIR}/delta_FR_vs_erm.csv", index=False)
import zipfile
with zipfile.ZipFile(f"{OUT_DIR}/consistency_tables.zip", "w") as z:
    for f in glob.glob(f"{OUT_DIR}/*.csv"): z.write(f, os.path.basename(f))
print("\nsend me:", f"{OUT_DIR}/consistency_tables.zip")